In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

In [2]:
coeffs = {
    "beta0": 2.0, "betaN": 0.35, "betaI": 0.50, "betaP": 0.25, "betaH": 0.05,
    "betaNN": -0.02, "betaII": -0.03, "betaPP": -0.01, "betaHH": -0.005,
    "betaNI": 0.04, "betaNP": 0.02, "betaNH": 0.01, "betaIP": 0.03, "betaIH": 0.01,
    "betaPH": 0.005
}
sigma = 0.05

def translog_yield(N, I, P, H, coeffs, sigma=0.05):
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    lnY_pred = (c["beta0"] +
                c["betaN"]*lnN + c["betaI"]*lnI + c["betaP"]*lnP + c["betaH"]*lnH +
                0.5*(c["betaNN"]*lnN**2 + c["betaII"]*lnI**2 +
                     c["betaPP"]*lnP**2 + c["betaHH"]*lnH**2) +
                c["betaNI"]*lnN*lnI + c["betaNP"]*lnN*lnP + c["betaNH"]*lnN*lnH +
                c["betaIP"]*lnI*lnP + c["betaIH"]*lnI*lnH + c["betaPH"]*lnP*lnH)

    lnY_actual = lnY_pred + np.random.normal(0, sigma, size=N.shape)

    return np.exp(lnY_actual)

In [3]:
np.random.seed(42)
n_obs = 1000
N = np.random.normal(240, 30, n_obs)
I = np.random.normal(320, 80, n_obs)
P = np.random.normal(210, 40, n_obs)
H = np.random.normal(4000, 500, n_obs)

Y = translog_yield(N, I, P, H, coeffs, sigma)

df = pd.DataFrame({"Nitrogen": N, "Irrigation": I, "Precipitation": P,
                   "Herbicide": H, "Yield": Y})

In [4]:
def calculate_log_contributions(N, I, P, H, coeffs):
    """Calculates the additive contribution of each feature to the log-yield."""
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    base_contribution = c["beta0"]

    contrib_N = (c["betaN"] * lnN +
                 0.5 * c["betaNN"] * lnN**2 +
                 0.5 * c["betaNI"] * lnN * lnI +
                 0.5 * c["betaNP"] * lnN * lnP +
                 0.5 * c["betaNH"] * lnN * lnH)

    contrib_I = (c["betaI"] * lnI +
                 0.5 * c["betaII"] * lnI**2 +
                 0.5 * c["betaNI"] * lnN * lnI +
                 0.5 * c["betaIP"] * lnI * lnP +
                 0.5 * c["betaIH"] * lnI * lnH)

    contrib_P = (c["betaP"] * lnP +
                 0.5 * c["betaPP"] * lnP**2 +
                 0.5 * c["betaNP"] * lnN * lnP +
                 0.5 * c["betaIP"] * lnI * lnP +
                 0.5 * c["betaPH"] * lnP * lnH)

    contrib_H = (c["betaH"] * lnH +
                 0.5 * c["betaHH"] * lnH**2 +
                 0.5 * c["betaNH"] * lnN * lnH +
                 0.5 * c["betaIH"] * lnI * lnH +
                 0.5 * c["betaPH"] * lnP * lnH)

    return base_contribution, contrib_N, contrib_I, contrib_P, contrib_H

contributions = df.apply(
    lambda row: calculate_log_contributions(
        row["Nitrogen"], row["Irrigation"], row["Precipitation"], row["Herbicide"], coeffs
    ),
    axis=1,
    result_type="expand"
)

df[['gt_base', 'gt_contrib_N', 'gt_contrib_I', 'gt_contrib_P', 'gt_contrib_H']] = contributions

In [5]:
def get_predicted_log_yield(N, I, P, H, coeffs):
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    lnY_pred = (c["beta0"] +
                c["betaN"]*lnN + c["betaI"]*lnI + c["betaP"]*lnP + c["betaH"]*lnH +
                0.5*(c["betaNN"]*lnN**2 + c["betaII"]*lnI**2 +
                     c["betaPP"]*lnP**2 + c["betaHH"]*lnH**2) +
                c["betaNI"]*lnN*lnI + c["betaNP"]*lnN*lnP + c["betaNH"]*lnN*lnH +
                c["betaIP"]*lnI*lnP + c["betaIH"]*lnI*lnH + c["betaPH"]*lnP*lnH)
    return lnY_pred

df['predicted_log_yield'] = get_predicted_log_yield(df['Nitrogen'], df['Irrigation'],
                                                    df['Precipitation'], df['Herbicide'],
                                                    coeffs)

df['sum_of_contributions'] = df['gt_base'] + df['gt_contrib_N'] + df['gt_contrib_I'] + df['gt_contrib_P'] + df['gt_contrib_H']

pd.set_option('display.width', 120)
print(df[['Yield', 'predicted_log_yield', 'sum_of_contributions']].head())

print("\nVerification: Are the sums close to the actual predicted log-yield?")
print(np.allclose(df['predicted_log_yield'], df['sum_of_contributions']))

          Yield  predicted_log_yield  sum_of_contributions
0  97596.843941            11.531775             11.531775
1  97610.435034            11.490300             11.490300
2  84178.656530            11.339796             11.339796
3  86461.500776            11.343823             11.343823
4  71875.136726            11.251029             11.251029

Verification: Are the sums close to the actual predicted log-yield?
True


In [6]:
df.head()

,Nitrogen,Irrigation,Precipitation,Herbicide,Yield,gt_base,gt_contrib_N,gt_contrib_I,gt_contrib_P,gt_contrib_H,predicted_log_yield,sum_of_contributions
0,254.901425,431.948435,182.992869,3046.096221,97596.843941,2.0,2.815651,3.871839,2.033977,0.810308,11.531775,11.531775
1,235.852071,393.970695,204.219253,3569.807495,97610.435034,2.0,2.780699,3.826665,2.064544,0.818393,11.490300,11.490300
2,259.430656,324.770430,178.303203,3793.197233,84178.656530,2.0,2.796571,3.720743,2.006095,0.816386,11.339796,11.339796
3,285.690896,268.245058,197.681539,4943.843829,86461.500776,2.0,2.831329,3.640602,2.036733,0.835160,11.343823,11.343823
4,232.975399,375.857865,134.255413,4278.276562,71875.136726,2.0,2.752061,3.767321,1.910175,0.821472,11.251029,11.251029


In [7]:
num_classes = 3
class_labels = ['Low', 'Medium', 'High']

df['Yield_Class'] = pd.qcut(df['Yield'], q=num_classes, labels=class_labels)

print("Class Distribution:")
print(df['Yield_Class'].value_counts())

Class Distribution:
Yield_Class
Low       334
High      334
Medium    332
Name: count, dtype: int64


In [ ]:
gt_cols = ['gt_contrib_N', 'gt_contrib_I', 'gt_contrib_P', 'gt_contrib_H']

global_ground_truth_importance = df[gt_cols].abs().mean().sort_values(ascending=False)

print("--- Global Ground-Truth Feature Importance ---")
print(global_ground_truth_importance)

--- Global Ground-Truth Feature Importance ---
gt_contrib_I    3.706509
gt_contrib_N    2.764945
gt_contrib_P    2.052278
gt_contrib_H    0.817947
dtype: float64


In [9]:
def calculate_log_contributions_shapley_split(N, I, P, H, coeffs):
    """
    Uses Shapley value logic to split interaction terms.
    Each feature gets credit for its marginal contribution averaged over orderings.
    """
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    base_contribution = c["beta0"]

    # Main effects (unambiguous)
    contrib_N_main = c["betaN"] * lnN + 0.5 * c["betaNN"] * lnN**2
    contrib_I_main = c["betaI"] * lnI + 0.5 * c["betaII"] * lnI**2
    contrib_P_main = c["betaP"] * lnP + 0.5 * c["betaPP"] * lnP**2
    contrib_H_main = c["betaH"] * lnH + 0.5 * c["betaHH"] * lnH**2

    # Interaction terms - Shapley split
    # For 2-way interaction: each feature gets 50% (same as equal split in this case)
    interaction_NI = c["betaNI"] * lnN * lnI
    interaction_NP = c["betaNP"] * lnN * lnP
    interaction_NH = c["betaNH"] * lnN * lnH
    interaction_IP = c["betaIP"] * lnI * lnP
    interaction_IH = c["betaIH"] * lnI * lnH
    interaction_PH = c["betaPH"] * lnP * lnH

    # Shapley allocation for 2-way interactions is 0.5 each
    contrib_N_shapley = contrib_N_main + 0.5 * (interaction_NI + interaction_NP + interaction_NH)
    contrib_I_shapley = contrib_I_main + 0.5 * (interaction_NI + interaction_IP + interaction_IH)
    contrib_P_shapley = contrib_P_main + 0.5 * (interaction_NP + interaction_IP + interaction_PH)
    contrib_H_shapley = contrib_H_main + 0.5 * (interaction_NH + interaction_IH + interaction_PH)

    return base_contribution, contrib_N_shapley, contrib_I_shapley, contrib_P_shapley, contrib_H_shapley


def calculate_log_contributions_proportional(N, I, P, H, coeffs):
    """
    Splits interaction terms proportionally to main effect magnitudes.
    Intuition: Features with larger main effects "deserve" more credit for interactions.
    """
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    base_contribution = c["beta0"]

    # Main effects
    main_N = abs(c["betaN"] * lnN + 0.5 * c["betaNN"] * lnN**2)
    main_I = abs(c["betaI"] * lnI + 0.5 * c["betaII"] * lnI**2)
    main_P = abs(c["betaP"] * lnP + 0.5 * c["betaPP"] * lnP**2)
    main_H = abs(c["betaH"] * lnH + 0.5 * c["betaHH"] * lnH**2)

    # Interaction terms
    interaction_NI = c["betaNI"] * lnN * lnI
    interaction_NP = c["betaNP"] * lnN * lnP
    interaction_NH = c["betaNH"] * lnN * lnH
    interaction_IP = c["betaIP"] * lnI * lnP
    interaction_IH = c["betaIH"] * lnI * lnH
    interaction_PH = c["betaPH"] * lnP * lnH

    # Proportional splits
    def proportional_split(val, weight_a, weight_b):
        total_weight = weight_a + weight_b
        if total_weight == 0:
            return 0.5 * val, 0.5 * val
        return (weight_a / total_weight) * val, (weight_b / total_weight) * val

    NI_to_N, NI_to_I = proportional_split(interaction_NI, main_N, main_I)
    NP_to_N, NP_to_P = proportional_split(interaction_NP, main_N, main_P)
    NH_to_N, NH_to_H = proportional_split(interaction_NH, main_N, main_H)
    IP_to_I, IP_to_P = proportional_split(interaction_IP, main_I, main_P)
    IH_to_I, IH_to_H = proportional_split(interaction_IH, main_I, main_H)
    PH_to_P, PH_to_H = proportional_split(interaction_PH, main_P, main_H)

    contrib_N_prop = c["betaN"] * lnN + 0.5 * c["betaNN"] * lnN**2 + NI_to_N + NP_to_N + NH_to_N
    contrib_I_prop = c["betaI"] * lnI + 0.5 * c["betaII"] * lnI**2 + NI_to_I + IP_to_I + IH_to_I
    contrib_P_prop = c["betaP"] * lnP + 0.5 * c["betaPP"] * lnP**2 + NP_to_P + IP_to_P + PH_to_P
    contrib_H_prop = c["betaH"] * lnH + 0.5 * c["betaHH"] * lnH**2 + NH_to_H + IH_to_H + PH_to_H

    return base_contribution, contrib_N_prop, contrib_I_prop, contrib_P_prop, contrib_H_prop


def calculate_log_contributions_main_only(N, I, P, H, coeffs):
    """
    Only considers main effects, ignores interactions entirely.
    Most conservative ground truth.
    """
    lnN, lnI, lnP, lnH = np.log(N), np.log(I), np.log(P), np.log(H)
    c = coeffs

    base_contribution = c["beta0"]

    contrib_N = c["betaN"] * lnN + 0.5 * c["betaNN"] * lnN**2
    contrib_I = c["betaI"] * lnI + 0.5 * c["betaII"] * lnI**2
    contrib_P = c["betaP"] * lnP + 0.5 * c["betaPP"] * lnP**2
    contrib_H = c["betaH"] * lnH + 0.5 * c["betaHH"] * lnH**2

    return base_contribution, contrib_N, contrib_I, contrib_P, contrib_H


# Apply all methods
methods = {
    'equal': calculate_log_contributions,
    'shapley': calculate_log_contributions_shapley_split,
    'proportional': calculate_log_contributions_proportional,
    'main_only': calculate_log_contributions_main_only
}

for method_name, method_func in methods.items():
    contributions = df.apply(
        lambda row: method_func(
            row["Nitrogen"], row["Irrigation"],
            row["Precipitation"], row["Herbicide"], coeffs
        ),
        axis=1,
        result_type="expand"
    )

    df[[f'gt_base_{method_name}',
        f'gt_contrib_N_{method_name}',
        f'gt_contrib_I_{method_name}',
        f'gt_contrib_P_{method_name}',
        f'gt_contrib_H_{method_name}']] = contributions

In [10]:
# Compare ground truths
print("\n" + "="*70)
print("COMPARING DIFFERENT GROUND TRUTH DEFINITIONS")
print("="*70)

for method_name in methods.keys():
    gt_cols = [f'gt_contrib_N_{method_name}',
               f'gt_contrib_I_{method_name}',
               f'gt_contrib_P_{method_name}',
               f'gt_contrib_H_{method_name}']

    importance = df[gt_cols].abs().mean().sort_values(ascending=False)
    importance.index = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

    print(f"\n{method_name.upper()} split:")
    print(importance)


COMPARING DIFFERENT GROUND TRUTH DEFINITIONS

EQUAL split:
Nitrogen         3.706509
Irrigation       2.764945
Precipitation    2.052278
Herbicide        0.817947
dtype: float64

SHAPLEY split:
Nitrogen         3.706509
Irrigation       2.764945
Precipitation    2.052278
Herbicide        0.817947
dtype: float64

PROPORTIONAL split:
Nitrogen         4.174135
Irrigation       2.856415
Precipitation    1.927916
Herbicide        0.383213
dtype: float64

MAIN_ONLY split:
Nitrogen         2.378583
Irrigation       1.616462
Precipitation    1.190226
Herbicide        0.242589
dtype: float64


In [ ]:
# Normalize for comparison
normalized_ground_truths = {}
for method_name in methods.keys():
    gt_cols = [f'gt_contrib_N_{method_name}',
               f'gt_contrib_I_{method_name}',
               f'gt_contrib_P_{method_name}',
               f'gt_contrib_H_{method_name}']

    importance = df[gt_cols].abs().mean()
    normalized_ground_truths[method_name] = importance / importance.sum()

# Create DataFrame for easy comparison
comparison_df = pd.DataFrame(normalized_ground_truths)
comparison_df.index = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

print("\n" + "="*70)
print("NORMALIZED COMPARISON (sum to 1)")
print("="*70)
print(comparison_df)

# Rank correlation between ground truths
from scipy.stats import spearmanr

print("\n" + "="*70)
print("RANK CORRELATIONS BETWEEN GROUND TRUTHS")
print("="*70)

for i, method1 in enumerate(methods.keys()):
    for method2 in list(methods.keys())[i+1:]:
        corr, _ = spearmanr(normalized_ground_truths[method1],
                           normalized_ground_truths[method2])
        print(f"{method1} vs {method2}: {corr:.4f}")

In [13]:
df_2 = df[['Yield_Class','Nitrogen','Irrigation','Precipitation','Herbicide']].copy()

In [20]:
df_2.head()

,Yield_Class,Nitrogen,Irrigation,Precipitation,Herbicide
0,High,254.901425,431.948435,182.992869,3046.096221
1,High,235.852071,393.970695,204.219253,3569.807495
2,Medium,259.430656,324.770430,178.303203,3793.197233
3,Medium,285.690896,268.245058,197.681539,4943.843829
4,Low,232.975399,375.857865,134.255413,4278.276562


In [15]:
feature_names = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

In [21]:
X = df_2[['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']]
y = df_2["Yield_Class"]

In [23]:
y = [0 if val == "Low" else 1 if val == "Medium" else 2 for val in y]

In [24]:
y

[2,
 2,
 1,
 1,
 0,
 1,
 2,
 1,
 2,
 1,
 1,
 0,
 2,
 0,
 2,
 1,
 0,
 0,
 0,
 0,
 2,
 0,
 1,
 2,
 0,
 1,
 1,
 2,
 2,
 0,
 1,
 0,
 1,
 1,
 1,
 2,
 0,
 0,
 1,
 1,
 2,
 1,
 1,
 1,
 0,
 2,
 2,
 0,
 2,
 2,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 2,
 2,
 1,
 2,
 0,
 1,
 0,
 1,
 0,
 0,
 2,
 1,
 2,
 2,
 2,
 1,
 0,
 0,
 1,
 0,
 1,
 2,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 2,
 1,
 0,
 2,
 1,
 2,
 2,
 0,
 2,
 0,
 2,
 0,
 1,
 2,
 1,
 0,
 2,
 1,
 1,
 1,
 2,
 1,
 0,
 2,
 0,
 1,
 2,
 1,
 2,
 1,
 2,
 0,
 0,
 2,
 2,
 2,
 2,
 0,
 2,
 1,
 0,
 1,
 1,
 2,
 1,
 1,
 2,
 2,
 0,
 2,
 2,
 0,
 1,
 0,
 1,
 2,
 0,
 1,
 1,
 1,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 2,
 1,
 1,
 2,
 2,
 1,
 0,
 2,
 0,
 0,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 1,
 1,
 1,
 0,
 2,
 0,
 2,
 1,
 1,
 2,
 2,
 1,
 0,
 0,
 2,
 0,
 1,
 1,
 0,
 0,
 0,
 2,
 0,
 0,
 1,
 1,
 1,
 0,
 2,
 2,
 1,
 2,
 1,
 0,
 1,
 1,
 0,
 0,
 2,
 0,
 1,
 1,
 2,
 0,
 2,
 0,
 1,
 1,
 2,
 0,
 0,
 2,
 1,
 0,
 2,
 2,
 0,
 1,
 2,
 0,
 0,
 0,
 2,
 0,
 2,
 2,
 0,
 1,
 2,
 0,
 2,
 1,
 0,
 1,
 0,
 0,
 0,
 2,
 0,


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)

In [28]:
np.savez("datasets/yield_synthetic/yield_preprocessed.npz", X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, feature_names=feature_names)